In [47]:
# --- BLOCK 1: SETUP ---
import pandas as pd
import numpy as np
import sys
import os
import joblib

# Time-Series & Machine Learning
from prophet import Prophet
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

# Visualization
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Local Pipeline Config
sys.path.append('..') 

# Safety check: Ensure our output folders exist before we save anything!
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

print("✅ Stage 1 Complete: Environment synchronized and directories ready.")

✅ Stage 1 Complete: Environment synchronized and directories ready.


In [48]:
# --- BLOCK 2: THE REAL SCIENCE DATA INGESTION (One Earth Fix) ---
import requests
from io import StringIO
import pandas as pd

print("📡 Accessing Real Global Climate Data...")

# 1. Load local real CO2 and GHG data
co2_df = pd.read_csv('../data/raw/owid-co2-data.csv') 
ghg_df = pd.read_csv('../data/raw/total-ghg-emissions.csv') 

# 2. Fetch REAL TEMPERATURE DATA (Bypassing OWID's Bot Protection)
temp_url = "https://ourworldindata.org/grapher/annual-temperature-anomalies.csv?v=1&csvType=full&useColumnShortNames=false"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}
response = requests.get(temp_url, headers=headers)

if response.status_code == 200:
    temp_df = pd.read_csv(StringIO(response.text))
else:
    raise Exception(f"❌ Server blocked the request! Status Code: {response.status_code}")

# 3. Rename columns
temp_df = temp_df.rename(columns={
    'Entity': 'country', 
    'Code': 'iso_code', 
    'Year': 'year', 
    'Temperature anomaly': 'Temp_Anomaly'
})
ghg_df = ghg_df.rename(columns={
    'Entity': 'country',
    'Code': 'iso_code',
    'Year': 'year',
    'Annual greenhouse gas emissions including land use': 'Total_GHG'
})

# 4. Merge the Science Data Together
df = pd.merge(co2_df[['country', 'iso_code', 'year', 'population', 'co2']], 
              ghg_df[['iso_code', 'year', 'Total_GHG']], 
              on=['iso_code', 'year'], how='left')

df = pd.merge(df, temp_df[['iso_code', 'year', 'Temp_Anomaly']], 
              on=['iso_code', 'year'], how='inner')

# ---------------------------------------------------------
# 🚀 THE "ONE EARTH" FIX: Remove Aggregate Regions 
# If we don't do this, the AI thinks Earth has 35 Billion people!
# ---------------------------------------------------------
df = df.dropna(subset=['iso_code'])
df = df[~df['iso_code'].str.startswith('OWID_')]

# 5. Rename for our Web App standard
df = df.rename(columns={
    'country': 'Real_Country_Name',
    'year': 'Year',
    'population': 'Population',
    'co2': 'CO2_Emissions'
})

# 6. Cleanup & Baselines
df['Total_GHG'] = df['Total_GHG'].fillna(0)
df['CO2_Emissions'] = df['CO2_Emissions'].fillna(0)
df['Population'] = df['Population'].fillna(0)

df['Average_Temperature'] = 14.0 + df['Temp_Anomaly'] 

print(f"✅ Stage 2 Complete: Clean 'One Earth' Data Loaded! {len(df)} records merged.")

📡 Accessing Real Global Climate Data...
✅ Stage 2 Complete: Clean 'One Earth' Data Loaded! 15725 records merged.


In [49]:
# --- BLOCK 3: ENGINEERING (The Physics Upgrade) ---
print("⚙️ Engineering Physics Features...")

# 1. MOVING AVERAGES (Smoothing out the weather noise to see real climate trends)
df['Temp_Moving_Avg'] = df.groupby('Real_Country_Name')['Temp_Anomaly'].transform(
    lambda x: x.rolling(window=10, min_periods=1).mean()
)

# 2. CLIMATE ZONES (Geographic Mapping)
CAPITAL_LAT = {
    "Canada": 45.4, "Brazil": -15.8, "Egypt": 30.0, "China": 39.9, 
    "USA": 38.9, "Russia": 55.8, "India": 28.6, "Australia": -35.3,
    "United States": 38.9 
}

def assign_zone(lat):
    lat = abs(lat)
    if lat >= 60: return 'Polar'
    elif lat >= 35: return 'Temperate'
    elif lat >= 23.5: return 'Subtropical'
    return 'Tropical'

df['climate_zone'] = df['Real_Country_Name'].map(CAPITAL_LAT).apply(
    lambda x: assign_zone(x) if pd.notna(x) else 'Unknown'
)

# 3. TIME DIMENSIONS
df['Decade'] = (df['Year'] // 10) * 10

# ---------------------------------------------------------
# 🌟 THE PHYSICS UPGRADE: CUMULATIVE CARBON 🌟
# Earth warms based on the TOTAL carbon in the atmosphere
# ---------------------------------------------------------
# We sort by year first to ensure the running total adds up perfectly
df = df.sort_values(by=['Real_Country_Name', 'Year'])
df['Cumulative_CO2'] = df.groupby('Real_Country_Name')['CO2_Emissions'].cumsum()

# 4. EXPORT THE PURE MASTER FILE
df.to_csv('../data/processed/supreme_dataset.csv', index=False)

print("✅ Stage 3 Complete: Physics Upgrade (Cumulative CO2) injected and saved!")

⚙️ Engineering Physics Features...
✅ Stage 3 Complete: Physics Upgrade (Cumulative CO2) injected and saved!


In [50]:
# --- BLOCK 4: THE GLOBAL AI WORKSHOP (True Extrapolation Upgrade) ---
import joblib
import pandas as pd
from sklearn.linear_model import Ridge # 🚀 THE MAGIC BULLET
from sklearn.preprocessing import StandardScaler
from prophet import Prophet
import json
from prophet.serialize import model_to_json

print("🧠 Initializing the AI Workshop...")

# 1. CREATE GLOBAL DATA
global_df = df.groupby('Year').agg({
    'CO2_Emissions': 'sum',
    'Population': 'sum',
    'Temp_Anomaly': 'mean', 
    'Cumulative_CO2': 'sum'
}).reset_index()

# 🌊 OCEAN INERTIA (10-YEAR LAG)
global_df['Target_Temp_Anomaly'] = global_df['Temp_Anomaly'].shift(-10)
train_df = global_df.dropna(subset=['Target_Temp_Anomaly'])

# 2. PREPARE THE TRAINING DATA
features = ['Year', 'CO2_Emissions', 'Cumulative_CO2', 'Population']
X = train_df[features]
y = train_df['Target_Temp_Anomaly']

# 3. SCALE THE BRAIN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. TRAIN THE LINEAR PHYSICS ENGINE
print("⚡ Training the Physics-Aware Linear Extrapolator...")
# Ridge Regression guarantees a perfect, infinite physical slope into the extreme danger zones!
ai_model = Ridge(alpha=1.0) 
ai_model.fit(X_scaled, y)

# 5. SAVE THE RECALIBRATED MODELS
joblib.dump(ai_model, '../models/supreme_nn_model.pkl') # Kept name so app.py stays unbroken
joblib.dump(scaler, '../models/supreme_scaler.pkl')
print("✅ Global AI trained and saved.")

# 6. PROPHET TIME-SERIES
global_df = global_df.sort_values("Year")
prophet_df = global_df[['Year', 'Temp_Anomaly']].copy()
prophet_df.rename(columns={'Year': 'ds', 'Temp_Anomaly': 'y'}, inplace=True)
prophet_df['ds'] = pd.to_datetime(prophet_df['ds'], format='%Y')

m = Prophet(yearly_seasonality=False, interval_width=0.95)
m.fit(prophet_df)

with open('../models/prophet_model.json', 'w') as fout:
    json.dump(model_to_json(m), fout)

future = m.make_future_dataframe(periods=20, freq='YE')
forecast = m.predict(future)
forecast.to_csv("../data/processed/future_forecast.csv", index=False)
print("✅ Forecast saved.")

# 7. FEATURE IMPORTANCE (Quick mapping for the UI chart)
from sklearn.ensemble import RandomForestRegressor
rf_explainer = RandomForestRegressor(random_state=42)
rf_explainer.fit(X, y)
importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": rf_explainer.feature_importances_
})
importance_df.to_csv("../data/processed/feature_importance.csv", index=False)
print("🎉 PIPELINE COMPLETE! The Extrapolation Engine is ready.")

06:15:29 - cmdstanpy - INFO - Chain [1] start processing


🧠 Initializing the AI Workshop...
⚡ Training the Physics-Aware Linear Extrapolator...
✅ Global AI trained and saved.


06:15:29 - cmdstanpy - INFO - Chain [1] done processing


✅ Forecast saved.
🎉 PIPELINE COMPLETE! The Extrapolation Engine is ready.
